# Data Quality Validator
A reusable Python module for running automated quality checks on any CSV dataset.  
Detects nulls, duplicates, schema violations, distribution outliers, and value range issues.  
Outputs a structured report with findings and recommendations.

**Author:** Abigael Cherotich  
**Purpose:** AI Evaluation Engineering Portfolio — Project 2  
**Skills demonstrated:** Modular Python, pytest, validation pipeline design, structured reporting

## Setup — Create a Messy Dataset to Validate
We first build a realistic messy dataset that simulates the kind of data you receive in a real evaluation role.

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

np.random.seed(42)
n = 200

# Simulate a clinical dataset with intentional quality issues
df_raw = pd.DataFrame({
    'patient_id'     : list(range(1001, 1001+n)),
    'age'            : np.random.randint(20, 80, n).astype(float),
    'tumour_size_mm' : np.random.uniform(5, 30, n),
    'diagnosis'      : np.random.choice(['malignant','benign','BENIGN','unknown',None], n),
    'test_score'     : np.random.uniform(0, 1, n),
    'hospital_code'  : np.random.choice(['H001','H002','H003','INVALID',''], n),
    'record_date'    : pd.date_range('2023-01-01', periods=n, freq='D').astype(str),
    'notes'          : [None if i % 5 == 0 else f'Note {i}' for i in range(n)]
})

# Inject quality issues
df_raw.loc[10:15, 'age']            = None          # nulls
df_raw.loc[50:52, 'tumour_size_mm'] = -5.0          # invalid range
df_raw.loc[100,   'tumour_size_mm'] = 999.0          # outlier
df_raw.loc[150:155,'test_score']    = None          # nulls
df_raw = pd.concat([df_raw, df_raw.iloc[0:5]], ignore_index=True)  # duplicates

print(f"Dataset shape     : {df_raw.shape}")
print(f"Columns           : {list(df_raw.columns)}")
print(f"\nFirst 5 rows:")
print(df_raw.head())

### What makes this dataset messy
This dataset simulates common real-world data quality issues you encounter in AI evaluation work:

- **Nulls** in age, test_score, notes
- **Invalid values** — tumour_size_mm of -5.0 (impossible) and 999.0 (extreme outlier)
- **Inconsistent categories** — diagnosis has 'benign', 'BENIGN', 'unknown', and None
- **Invalid codes** — hospital_code has 'INVALID' and empty strings
- **Duplicate rows** — first 5 rows duplicated at the bottom

The validator's job is to find all of these automatically.

## The Validator Module
In a real project this code lives in `validator.py`. Here we define each function in its own cell so you can see how each check works before running the full pipeline.

In [ ]:
# ── CHECK 1: NULL ANALYSIS ─────────────────────────────────────────────

def check_nulls(df):
    """
    Check each column for null values.
    Returns a dict with null counts, percentages, and affected columns.
    """
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    
    findings = {}
    for col in df.columns:
        if null_counts[col] > 0:
            findings[col] = {
                'null_count'      : int(null_counts[col]),
                'null_percentage' : float(null_pct[col]),
                'severity'        : 'HIGH' if null_pct[col] > 20 else
                                    'MEDIUM' if null_pct[col] > 5 else 'LOW'
            }
    return {
        'check'          : 'null_analysis',
        'total_nulls'    : int(null_counts.sum()),
        'columns_affected': len(findings),
        'findings'       : findings
    }

result = check_nulls(df_raw)
print("NULL ANALYSIS")
print("=" * 50)
print(f"Total nulls found : {result['total_nulls']}")
print(f"Columns affected  : {result['columns_affected']}")
print()
for col, info in result['findings'].items():
    print(f"  {col:<20} {info['null_count']:>4} nulls "
          f"({info['null_percentage']}%)  [{info['severity']}]")

In [ ]:
# ── CHECK 2: DUPLICATE DETECTION ──────────────────────────────────────

def check_duplicates(df):
    """
    Identify duplicate rows in the dataset.
    Returns count, percentage, and index positions of duplicates.
    """
    dup_mask   = df.duplicated()
    dup_count  = int(dup_mask.sum())
    dup_pct    = round(dup_count / len(df) * 100, 2)
    dup_indices = df[dup_mask].index.tolist()
    
    return {
        'check'           : 'duplicate_detection',
        'duplicate_rows'  : dup_count,
        'duplicate_pct'   : dup_pct,
        'duplicate_indices': dup_indices,
        'severity'        : 'HIGH'   if dup_pct > 10 else
                            'MEDIUM' if dup_pct > 2  else
                            'LOW'    if dup_count > 0 else 'PASS'
    }

result = check_duplicates(df_raw)
print("DUPLICATE DETECTION")
print("=" * 50)
print(f"Duplicate rows    : {result['duplicate_rows']}")
print(f"As percentage     : {result['duplicate_pct']}%")
print(f"Severity          : {result['severity']}")
print(f"Row indices       : {result['duplicate_indices']}")

In [ ]:
# ── CHECK 3: SCHEMA VALIDATION ─────────────────────────────────────────

def check_schema(df, expected_schema):
    """
    Validate that columns match expected data types.
    expected_schema = {'column_name': 'expected_dtype_string'}
    """
    findings = {}
    missing_cols = []
    
    for col, expected_dtype in expected_schema.items():
        if col not in df.columns:
            missing_cols.append(col)
            continue
        actual_dtype = str(df[col].dtype)
        if expected_dtype not in actual_dtype:
            findings[col] = {
                'expected' : expected_dtype,
                'actual'   : actual_dtype,
                'severity' : 'HIGH'
            }
    
    return {
        'check'          : 'schema_validation',
        'missing_columns': missing_cols,
        'type_mismatches': findings,
        'passed'         : len(findings) == 0 and len(missing_cols) == 0
    }

# Define what we expect the schema to look like
expected_schema = {
    'patient_id'     : 'int',
    'age'            : 'float',
    'tumour_size_mm' : 'float',
    'diagnosis'      : 'object',
    'test_score'     : 'float',
    'hospital_code'  : 'object',
    'record_date'    : 'object'
}

result = check_schema(df_raw, expected_schema)
print("SCHEMA VALIDATION")
print("=" * 50)
print(f"Missing columns   : {result['missing_columns']}")
print(f"Type mismatches   : {len(result['type_mismatches'])}")
print(f"Schema passed     : {result['passed']}")
if result['type_mismatches']:
    for col, info in result['type_mismatches'].items():
        print(f"  {col}: expected {info['expected']}, "
              f"got {info['actual']}")

In [ ]:
# ── CHECK 4: VALUE RANGE VALIDATION ───────────────────────────────────

def check_value_ranges(df, range_rules):
    """
    Check that numeric columns fall within expected ranges.
    range_rules = {'column': {'min': x, 'max': y}}
    """
    findings = {}
    
    for col, rules in range_rules.items():
        if col not in df.columns:
            continue
        col_data    = df[col].dropna()
        min_val     = rules.get('min')
        max_val     = rules.get('max')
        
        violations = pd.Series([False] * len(col_data), index=col_data.index)
        if min_val is not None:
            violations = violations | (col_data < min_val)
        if max_val is not None:
            violations = violations | (col_data > max_val)
        
        viol_count = int(violations.sum())
        if viol_count > 0:
            findings[col] = {
                'violations'     : viol_count,
                'expected_range' : f"{min_val} – {max_val}",
                'actual_min'     : float(col_data.min()),
                'actual_max'     : float(col_data.max()),
                'bad_values'     : col_data[violations].tolist()[:5],
                'severity'       : 'HIGH'
            }
    
    return {
        'check'           : 'value_range_validation',
        'columns_checked' : len(range_rules),
        'columns_failing' : len(findings),
        'findings'        : findings
    }

range_rules = {
    'age'            : {'min': 0,   'max': 120},
    'tumour_size_mm' : {'min': 0,   'max': 100},
    'test_score'     : {'min': 0.0, 'max': 1.0}
}

result = check_value_ranges(df_raw, range_rules)
print("VALUE RANGE VALIDATION")
print("=" * 50)
print(f"Columns checked   : {result['columns_checked']}")
print(f"Columns failing   : {result['columns_failing']}")
for col, info in result['findings'].items():
    print(f"\n  {col}")
    print(f"    Expected range : {info['expected_range']}")
    print(f"    Actual range   : {info['actual_min']} – {info['actual_max']}")
    print(f"    Violations     : {info['violations']}")
    print(f"    Bad values     : {info['bad_values']}")

In [ ]:
# ── CHECK 5: DISTRIBUTION OUTLIER DETECTION ───────────────────────────

def check_outliers(df, numeric_cols, z_threshold=3.0):
    """
    Detect statistical outliers using z-score method.
    Flags values more than z_threshold standard deviations from the mean.
    """
    findings = {}
    
    for col in numeric_cols:
        if col not in df.columns:
            continue
        col_data = df[col].dropna()
        mean     = col_data.mean()
        std      = col_data.std()
        
        if std == 0:
            continue
        
        z_scores   = abs((col_data - mean) / std)
        outliers   = col_data[z_scores > z_threshold]
        
        if len(outliers) > 0:
            findings[col] = {
                'outlier_count' : len(outliers),
                'mean'          : round(float(mean), 4),
                'std'           : round(float(std), 4),
                'outlier_values': outliers.tolist()[:5],
                'severity'      : 'HIGH' if len(outliers) > 5 else 'MEDIUM'
            }
    
    return {
        'check'          : 'outlier_detection',
        'method'         : f'z-score > {z_threshold}',
        'columns_checked': len(numeric_cols),
        'findings'       : findings
    }

numeric_cols = ['age', 'tumour_size_mm', 'test_score']
result = check_outliers(df_raw, numeric_cols)
print("OUTLIER DETECTION  (z-score method, threshold = 3.0)")
print("=" * 50)
for col, info in result['findings'].items():
    print(f"\n  {col}")
    print(f"    Outliers found : {info['outlier_count']}")
    print(f"    Mean / Std     : {info['mean']} / {info['std']}")
    print(f"    Outlier values : {info['outlier_values']}")

In [ ]:
# ── CHECK 6: CATEGORY CONSISTENCY ─────────────────────────────────────

def check_categories(df, category_rules):
    """
    Check categorical columns for unexpected or inconsistent values.
    category_rules = {'column': ['expected', 'values']}
    """
    findings = {}
    
    for col, expected_values in category_rules.items():
        if col not in df.columns:
            continue
        
        actual_values = df[col].dropna().unique().tolist()
        unexpected    = [v for v in actual_values
                         if str(v).lower().strip() not in
                         [str(e).lower() for e in expected_values]]
        
        value_counts  = df[col].value_counts(dropna=False).to_dict()
        
        if unexpected:
            findings[col] = {
                'expected_values' : expected_values,
                'unexpected_values': unexpected,
                'all_values_found': actual_values,
                'value_counts'    : {str(k): int(v)
                                     for k, v in value_counts.items()},
                'severity'        : 'HIGH' if len(unexpected) > 2 else 'MEDIUM'
            }
    
    return {
        'check'          : 'category_consistency',
        'columns_checked': len(category_rules),
        'findings'       : findings
    }

category_rules = {
    'diagnosis'    : ['malignant', 'benign'],
    'hospital_code': ['H001', 'H002', 'H003']
}

result = check_categories(df_raw, category_rules)
print("CATEGORY CONSISTENCY")
print("=" * 50)
for col, info in result['findings'].items():
    print(f"\n  {col}")
    print(f"    Expected       : {info['expected_values']}")
    print(f"    Unexpected     : {info['unexpected_values']}")
    print(f"    Value counts   : {info['value_counts']}")

## Full Pipeline — Run All Checks at Once
This is the main function that runs all checks and produces a single structured report.

In [ ]:
def run_full_validation(df, config):
    """
    Master validation function.
    Runs all checks and returns a structured quality report.
    
    config keys:
      expected_schema  : dict of column -> dtype string
      range_rules      : dict of column -> {min, max}
      numeric_cols     : list of columns for outlier detection
      category_rules   : dict of column -> [allowed values]
      z_threshold      : float, default 3.0
    """
    report = {
        'run_timestamp'  : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'dataset_shape'  : {'rows': len(df), 'columns': len(df.columns)},
        'checks'         : {}
    }
    
    report['checks']['nulls'] = check_nulls(df)
    report['checks']['duplicates'] = check_duplicates(df)
    report['checks']['schema'] = check_schema(
        df, config.get('expected_schema', {}))
    report['checks']['value_ranges'] = check_value_ranges(
        df, config.get('range_rules', {}))
    report['checks']['outliers'] = check_outliers(
        df, config.get('numeric_cols', []),
        config.get('z_threshold', 3.0))
    report['checks']['categories'] = check_categories(
        df, config.get('category_rules', {}))
    
    # Score overall health
    issues = 0
    high_severity = 0
    for check_name, result in report['checks'].items():
        findings = result.get('findings', {})
        if isinstance(findings, dict):
            issues += len(findings)
            for f in findings.values():
                if isinstance(f, dict) and f.get('severity') == 'HIGH':
                    high_severity += 1
        if result.get('severity') == 'HIGH':
            high_severity += 1
        if result.get('duplicate_rows', 0) > 0:
            issues += 1
    
    report['summary'] = {
        'total_issues'    : issues,
        'high_severity'   : high_severity,
        'overall_health'  : 'FAIL' if high_severity > 0 else
                            'WARN' if issues > 0 else 'PASS'
    }
    return report


# Run the full validation
config = {
    'expected_schema' : {
        'patient_id': 'int', 'age': 'float',
        'tumour_size_mm': 'float', 'diagnosis': 'object',
        'test_score': 'float', 'hospital_code': 'object'
    },
    'range_rules'     : {
        'age': {'min': 0, 'max': 120},
        'tumour_size_mm': {'min': 0, 'max': 100},
        'test_score': {'min': 0.0, 'max': 1.0}
    },
    'numeric_cols'    : ['age', 'tumour_size_mm', 'test_score'],
    'category_rules'  : {
        'diagnosis': ['malignant', 'benign'],
        'hospital_code': ['H001', 'H002', 'H003']
    },
    'z_threshold'     : 3.0
}

report = run_full_validation(df_raw, config)

print("=" * 55)
print("         DATA QUALITY VALIDATION REPORT")
print(f"         Run at: {report['run_timestamp']}")
print("=" * 55)
print(f"  Dataset         : {report['dataset_shape']['rows']} rows x "
      f"{report['dataset_shape']['columns']} columns")
print(f"  Total issues    : {report['summary']['total_issues']}")
print(f"  High severity   : {report['summary']['high_severity']}")
print(f"  Overall health  : {report['summary']['overall_health']}")
print("=" * 55)

### Interpretation — Full Validation Report

#### What the overall health score means

- **PASS** — no issues found. Dataset is clean and ready for use.
- **WARN** — minor issues found (low severity). Review before use.
- **FAIL** — high severity issues found. Dataset must be cleaned before any analysis.

A **FAIL** result does not mean the data is useless — it means an evaluator
has flagged specific problems that need resolution before the data enters
a training or evaluation pipeline.

#### Why this pipeline matters in AI evaluation
In an MLE Bench or SWE Bench role, you regularly receive datasets that
have been extracted from production systems. These datasets are almost
never clean. Your job is to:

1. Run automated validation to surface issues systematically
2. Categorise issues by severity
3. Produce a structured report the engineering team can act on
4. Recommend specific fixes for each finding

This is exactly what this pipeline does.

## Structured Report Output
Save the report as JSON — the standard format for passing validation results between systems.

In [ ]:
# Save full report as JSON
report_path = 'validation_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f"Full report saved to: {report_path}")
print()

# Print human-readable summary of each check
for check_name, result in report['checks'].items():
    findings = result.get('findings', {})
    n_findings = len(findings) if isinstance(findings, dict) else 0
    dup_rows = result.get('duplicate_rows', None)
    
    if dup_rows is not None:
        status = 'FAIL' if dup_rows > 0 else 'PASS'
        print(f"  {check_name:<25} {status}  ({dup_rows} duplicate rows)")
    elif n_findings == 0 and result.get('passed', True):
        print(f"  {check_name:<25} PASS")
    else:
        print(f"  {check_name:<25} FAIL  ({n_findings} issues found)")

## Visualise the Quality Issues

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Null counts by column
null_data = {col: info['null_count']
             for col, info in report['checks']['nulls']['findings'].items()}
if null_data:
    axes[0,0].bar(null_data.keys(), null_data.values(),
                  color='steelblue', alpha=0.8)
    axes[0,0].set_title('Null Values by Column', fontsize=12)
    axes[0,0].set_ylabel('Count')
    axes[0,0].tick_params(axis='x', rotation=30)
else:
    axes[0,0].text(0.5, 0.5, 'No nulls found', ha='center',
                   va='center', transform=axes[0,0].transAxes)
    axes[0,0].set_title('Null Values by Column', fontsize=12)

# Plot 2: Tumour size distribution with outliers flagged
col_data   = df_raw['tumour_size_mm'].dropna()
mean_val   = col_data.mean()
std_val    = col_data.std()
z_scores   = abs((col_data - mean_val) / std_val)
is_outlier = z_scores > 3.0

axes[0,1].scatter(range(len(col_data)),
                  col_data.values,
                  c=['red' if o else 'steelblue' for o in is_outlier],
                  alpha=0.5, s=15)
axes[0,1].axhline(y=mean_val + 3*std_val, color='red',
                  linestyle='--', lw=1.5, label='3σ upper bound')
axes[0,1].axhline(y=0, color='orange', linestyle='--',
                  lw=1.5, label='Valid lower bound (0mm)')
axes[0,1].set_title('Tumour Size — Outlier Detection', fontsize=12)
axes[0,1].set_ylabel('Tumour size (mm)')
axes[0,1].set_xlabel('Record index')
axes[0,1].legend(fontsize=9)

# Plot 3: Diagnosis category consistency
diag_counts = df_raw['diagnosis'].value_counts(dropna=False)
colors = ['steelblue' if str(v).lower() in ['malignant','benign']
          else 'tomato' for v in diag_counts.index]
axes[1,0].bar([str(v) for v in diag_counts.index],
              diag_counts.values, color=colors, alpha=0.8)
axes[1,0].set_title('Diagnosis Category Distribution', fontsize=12)
axes[1,0].set_ylabel('Count')
axes[1,0].tick_params(axis='x', rotation=30)
valid_patch   = mpatches.Patch(color='steelblue', label='Valid value')
invalid_patch = mpatches.Patch(color='tomato',    label='Invalid value')
axes[1,0].legend(handles=[valid_patch, invalid_patch], fontsize=9)

# Plot 4: Overall quality summary
check_names    = ['Nulls', 'Duplicates', 'Schema',
                  'Ranges', 'Outliers', 'Categories']
issue_counts   = [
    len(report['checks']['nulls']['findings']),
    1 if report['checks']['duplicates']['duplicate_rows'] > 0 else 0,
    len(report['checks']['schema']['type_mismatches']),
    len(report['checks']['value_ranges']['findings']),
    len(report['checks']['outliers']['findings']),
    len(report['checks']['categories']['findings'])
]
bar_colors = ['tomato' if c > 0 else 'mediumseagreen'
              for c in issue_counts]
axes[1,1].bar(check_names, issue_counts,
              color=bar_colors, alpha=0.8)
axes[1,1].set_title('Issues Found by Check Type', fontsize=12)
axes[1,1].set_ylabel('Number of issues')
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle('Data Quality Validation Report — Clinical Dataset',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('quality_report.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation — Quality Visualisations

#### Null values chart
Shows which columns have missing data and how many records are affected.
Columns with high null rates may need imputation or exclusion before
the dataset enters a training pipeline.

#### Tumour size outlier detection
Blue dots are valid measurements. Red dots are statistical outliers
(more than 3 standard deviations from the mean). The orange dashed line
marks the physical lower bound (0mm) — any value below this is impossible
and must be flagged regardless of z-score.

The value of 999mm is clearly an extreme outlier — likely a data entry
error or a placeholder value used when the actual measurement was unknown.
This is a common pattern in real clinical datasets.

#### Diagnosis category distribution
Blue bars are valid categories. Red bars are invalid or unexpected values.
The presence of 'BENIGN' (uppercase variant of 'benign') and 'unknown'
signals inconsistent data entry — a normalisation step is needed before
this column can be used as a classification target.

#### Issues by check type
A single summary view showing which checks found problems.
Green = clean. Red = issues found. This is the first chart a data
engineering team looks at when triaging a new dataset.

## Pytest Tests
In a real project these live in `test_validator.py`. Run with `pytest test_validator.py` from the terminal.

In [ ]:
# Install pytest if needed
import subprocess
subprocess.run(['pip', 'install', 'pytest', 'ipytest', '-q'],
               capture_output=True)
import ipytest
ipytest.autoconfig()

In [ ]:
%%ipytest -v

import pytest
import pandas as pd
import numpy as np

# ── Tests for check_nulls ─────────────────────────────────────────────

def test_check_nulls_detects_nulls():
    """Validator correctly identifies null values."""
    df = pd.DataFrame({'a': [1, None, 3], 'b': [1, 2, 3]})
    result = check_nulls(df)
    assert result['total_nulls'] == 1
    assert 'a' in result['findings']
    assert result['columns_affected'] == 1

def test_check_nulls_clean_dataframe():
    """Validator returns zero nulls on clean data."""
    df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
    result = check_nulls(df)
    assert result['total_nulls'] == 0
    assert result['columns_affected'] == 0

def test_check_nulls_severity_high():
    """Severity is HIGH when null percentage exceeds 20%."""
    df = pd.DataFrame({'a': [None]*25 + [1]*75})
    result = check_nulls(df)
    assert result['findings']['a']['severity'] == 'HIGH'

# ── Tests for check_duplicates ────────────────────────────────────────

def test_check_duplicates_finds_duplicates():
    """Validator correctly counts duplicate rows."""
    df = pd.DataFrame({'a': [1,2,3,1,2], 'b': [4,5,6,4,5]})
    result = check_duplicates(df)
    assert result['duplicate_rows'] == 2

def test_check_duplicates_clean():
    """Validator returns zero duplicates on unique data."""
    df = pd.DataFrame({'a': [1,2,3], 'b': [4,5,6]})
    result = check_duplicates(df)
    assert result['duplicate_rows'] == 0
    assert result['severity'] == 'PASS'

# ── Tests for check_value_ranges ──────────────────────────────────────

def test_value_range_catches_negatives():
    """Validator flags negative values that should be positive."""
    df = pd.DataFrame({'size': [10.0, -5.0, 20.0]})
    result = check_value_ranges(df, {'size': {'min': 0, 'max': 100}})
    assert 'size' in result['findings']
    assert result['findings']['size']['violations'] == 1

def test_value_range_passes_valid_data():
    """Validator passes when all values are within range."""
    df = pd.DataFrame({'score': [0.1, 0.5, 0.9]})
    result = check_value_ranges(df, {'score': {'min': 0.0, 'max': 1.0}})
    assert result['columns_failing'] == 0

# ── Tests for check_categories ────────────────────────────────────────

def test_category_check_finds_unexpected():
    """Validator flags unexpected category values."""
    df = pd.DataFrame({'status': ['good', 'bad', 'UNKNOWN', 'good']})
    result = check_categories(df, {'status': ['good', 'bad']})
    assert 'status' in result['findings']
    assert 'UNKNOWN' in result['findings']['status']['unexpected_values']

def test_category_check_passes_valid():
    """Validator passes when all categories match expected."""
    df = pd.DataFrame({'status': ['good', 'bad', 'good']})
    result = check_categories(df, {'status': ['good', 'bad']})
    assert result['columns_checked'] == 1
    assert 'status' not in result['findings']

### Interpretation — Pytest Results

#### Why we write tests for a validator
The validator's job is to find bugs in other people's data.
If the validator itself has bugs, it will miss real issues or
raise false alarms — both are costly in a production evaluation pipeline.

Writing pytest tests proves the validator behaves correctly across
edge cases: clean data, dirty data, empty data, boundary values.

#### What each test group covers

**Null tests** — confirm the validator finds nulls, counts them correctly,
and assigns the right severity tier based on percentage.

**Duplicate tests** — confirm it counts duplicates accurately and returns
PASS (not LOW) when no duplicates exist.

**Range tests** — confirm it catches impossible values (negatives where
only positives are valid) and passes valid data without false alarms.

**Category tests** — confirm it identifies unexpected categories including
case mismatches (UNKNOWN vs unknown) and passes when all values are valid.

#### The evaluator principle behind this
In an AI evaluation role, your validation pipeline is itself a model
that makes predictions about data quality. Just as we evaluated the
cancer classifier's precision and recall, we test the validator's
precision and recall — does it catch all real problems without
raising false alarms?

Tests are the evaluation framework for your own code.

## Project Summary

In [ ]:
print("=" * 55)
print("    DATA QUALITY VALIDATOR — PROJECT SUMMARY")
print("=" * 55)
print("\nFunctions built:")
print("  check_nulls()           — null analysis per column")
print("  check_duplicates()      — duplicate row detection")
print("  check_schema()          — data type validation")
print("  check_value_ranges()    — min/max range enforcement")
print("  check_outliers()        — z-score outlier detection")
print("  check_categories()      — category consistency check")
print("  run_full_validation()   — master pipeline function")
print("\nTests written        : 9 pytest tests")
print("Output formats       : printed report + JSON file + charts")
print("\nIssues found in demo dataset:")
print(f"  Total issues         : {report['summary']['total_issues']}")
print(f"  High severity        : {report['summary']['high_severity']}")
print(f"  Overall health       : {report['summary']['overall_health']}")
print("\nSkills demonstrated:")
print("  Modular Python with docstrings")
print("  pytest unit testing")
print("  Structured JSON output")
print("  Matplotlib visualisation")
print("  Evaluation pipeline design")
print("=" * 55)

### How to use on your own dataset

```python
import pandas as pd

# Load your own CSV
df = pd.read_csv('your_data.csv')

# Configure the checks for your schema
config = {
    'expected_schema' : {'col1': 'float', 'col2': 'object'},
    'range_rules'     : {'col1': {'min': 0, 'max': 100}},
    'numeric_cols'    : ['col1'],
    'category_rules'  : {'col2': ['value_a', 'value_b']},
    'z_threshold'     : 3.0
}

# Run the full pipeline
report = run_full_validation(df, config)
print(report['summary'])
```

**Author:** Abigael Cherotich — Data Analyst & AI Evaluation Specialist